In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
import scipy.io as sio
import copy
from scipy.stats import norm
from matplotlib import animation, rc
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.patches import Circle
from matplotlib.patches import Arrow
from matplotlib.collections import PatchCollection
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import sys

In [ ]:
# for the font of the plots
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

In [ ]:
# python version of MATLAB smooth function
# from https://stackoverflow.com/questions/40443020/matlabs-smooth-implementation-n-point-moving-average-in-numpy-python

def smooth(a,WSZ):
    # a: NumPy 1-D array containing the data to be smoothed
    # WSZ: smoothing window size needs, which must be odd number,
    # as in the original MATLAB implementation
    
    if len(a)<WSZ and len(a)%2==1:
        WSZ = len(a)
    elif len(a)<WSZ and len(a)%2==0:
        WSZ = len(a)-1
    
    out0 = np.convolve(a,np.ones(WSZ,dtype=int),'valid')/WSZ    
    r = np.arange(1,WSZ-1,2)
    start = np.cumsum(a[:WSZ-1])[::2]/r
    stop = (np.cumsum(a[:-WSZ:-1])[::2]/r)[::-1]
    return np.concatenate((  start , out0, stop  ))

# load data

In [ ]:
numx = 30
numy = 30
N = numx*numy
nrec = 400

base_dir = '\\directory_of_data'

file_name = 'xarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    xarr = np.fromfile(f,np.float64,sep=' ')
xarr = xarr.reshape((nrec,N))

file_name = 'yarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    yarr = np.fromfile(f,np.float64,sep=' ')
yarr = yarr.reshape((nrec,N))

file_name = 'tharr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    tharr = np.fromfile(f,np.float64,sep=' ')
tharr = tharr.reshape((nrec,N))

file_name = 'vxarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vxarr = np.fromfile(f,np.float64,sep=' ')
vxarr = vxarr.reshape((nrec,N))

file_name = 'vyarr.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vyarr = np.fromfile(f,np.float64,sep=' ')
vyarr = vyarr.reshape((nrec,N))

file_name = 'vxnoise.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vxnoise = np.fromfile(f,np.float64,sep=' ')
vxnoise = vxnoise.reshape((nrec,N))

file_name = 'vynoise.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    vynoise = np.fromfile(f,np.float64,sep=' ')
vynoise = vynoise.reshape((nrec,N))


# smooth the trajectory and get the deviation

In [ ]:
eqmPosSm=51 # smoothing factor to find mean/eqm position

numx = 30
numy = 30

xdev = np.zeros([nrec,(numx-2)*(numy-2)])
ydev = np.zeros([nrec,(numx-2)*(numy-2)])
xeqm = np.zeros([nrec,(numx-2)*(numy-2)])
yeqm = np.zeros([nrec,(numx-2)*(numy-2)])

vxdev = np.zeros([nrec,(numx-2)*(numy-2)])
vydev = np.zeros([nrec,(numx-2)*(numy-2)])
vxeqm = np.zeros([nrec,(numx-2)*(numy-2)])
vyeqm = np.zeros([nrec,(numx-2)*(numy-2)])

for i in range(1,numx-1):
    for j in range(1,numy-1):
        xi=xarr[:,j*numx+i] 
        idx = np.where(xi!=0)
        yi=yarr[:,j*numx+i]
    
        xdev[idx,(j-1)*(numx-2)+(i-1)]=xi[idx]-smooth(xi[idx],eqmPosSm)
        ydev[idx,(j-1)*(numx-2)+(i-1)]=yi[idx]-smooth(yi[idx],eqmPosSm)
        xeqm[idx,(j-1)*(numx-2)+(i-1)]=smooth(xi[idx],eqmPosSm)
        yeqm[idx,(j-1)*(numx-2)+(i-1)]=smooth(yi[idx],eqmPosSm)
    
        vxi=vxarr[:,j*numx+i] 
        vyi=vyarr[:,j*numx+i]
    
        vxdev[idx,(j-1)*(numx-2)+(i-1)]=vxi[idx]-smooth(vxi[idx],eqmPosSm)
        vydev[idx,(j-1)*(numx-2)+(i-1)]=vyi[idx]-smooth(vyi[idx],eqmPosSm)
        vxeqm[idx,(j-1)*(numx-2)+(i-1)]=smooth(vxi[idx],eqmPosSm)
        vyeqm[idx,(j-1)*(numx-2)+(i-1)]=smooth(vyi[idx],eqmPosSm)

In [ ]:
numx = 30
numy = 30
R = 0.5
spacing = 1.2
dx = 0.3*spacing
dy = 0.3*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R

#dt = 0.05
dt = 0.1
#dt = 10.0
#dt = 2.0

dv_list_all = []
dvx_list_all = []
dvy_list_all = []

trange = np.arange(10,nrec-1)

for i in range((numx-2)*(numy-2)):
    for j in trange:
        xcur = xeqm[j,i]
        xnext = xeqm[j+1,i]
        ycur = yeqm[j,i]
        ynext = yeqm[j+1,i]
        
        dxcur = xdev[j,i]
        dxnext = xdev[j+1,i]
        dycur = ydev[j,i]
        dynext = ydev[j+1,i]
        
        dvx = (dxnext-dxcur)/dt
        dvy = (dynext-dycur)/dt
            
        dv_list_all.append(np.sqrt(dvx**2+dvy**2))
        dvx_list_all.append(dvx)
        dvy_list_all.append(dvy)
        

# Gaussian fit of $\delta v$ distribution from the starfish embryo model simulation

In [ ]:
# need to try different xhalf and yhalf values to get the best Gaussian fit

dvx_sum = np.array(dvx_list)
dvy_sum = np.array(dvy_list)

mu_dvx,std_dvx = norm.fit(dvx_sum)
xhalf = 0.034
xval = np.linspace(-xhalf, xhalf, 100)
normfit_dvx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_sum)*2*xhalf/100

mu_dvy,std_dvy = norm.fit(dvy_sum)
yhalf = 0.032
yval = np.linspace(-yhalf, yhalf, 100)
normfit_dvy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_sum)*2*yhalf/100

%matplotlib inline

fig = plt.figure(figsize=(10,5),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,2,1)
ax1.hist(dvx_sum,bins=100)
ax1.plot(xval,normfit_dvx,'k')
ax1.text(-0.03,11000,"$\mu$=%s"%("%.4f"%(mu_dvx)),fontsize=15)
ax1.text(-0.03,10000,"$\sigma$=%s"%("%.4f"%(std_dvx)),fontsize=15)
ax1.text(-0.052,12000,'(a)',fontsize=20)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)

ax2 = fig.add_subplot(1,2,2)
ax2.hist(dvy_sum,bins=100)
ax2.plot(yval,normfit_dvy,'k')
ax2.text(-0.035,10500,"$\mu$=%s"%("%.4f"%(mu_dvy)),fontsize=15)
ax2.text(-0.035,9500,"$\sigma$=%s"%("%.4f"%(std_dvy)),fontsize=15)
ax2.text(-0.053,11500,'(b)',fontsize=20)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)

fig.tight_layout()
plt.show()


# $\delta v$ statistics at a certian time point

In [ ]:
numx = 30
numy = 30
R = 0.5
spacing = 1.2
dx = 0.3*spacing
dy = 0.3*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R

#dt = 0.05
dt = 0.1
#dt = 10.0
#dt = 2.0

dv_list = []
dvx_list = []
dvy_list = []

this_t = 120

for i in range((numx-2)*(numy-2)):
    xcur = xeqm[this_t,i]
    xnext = xeqm[this_t+1,i]
    ycur = yeqm[this_t,i]
    ynext = yeqm[this_t+1,i]
        
    dxcur = xdev[this_t,i]
    dxnext = xdev[this_t+1,i]
    dycur = ydev[this_t,i]
    dynext = ydev[this_t+1,i]
        
    dvx = (dxnext-dxcur)/dt
    dvy = (dynext-dycur)/dt
            
    dv_list.append(np.sqrt(dvx**2+dvy**2))
    dvx_list.append(dvx)
    dvy_list.append(dvy)
    

# Gaussian fit

In [ ]:
nBins = 30

dvx_sum = np.array(dvx_list)
dvy_sum = np.array(dvy_list)

mu_dvx,std_dvx = norm.fit(dvx_sum)
xhalf = 0.015
xval = np.linspace(-xhalf, xhalf, 100)
normfit_dvx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_sum)*2*xhalf/nBins

mu_dvy,std_dvy = norm.fit(dvy_sum)
yhalf = 0.015
yval = np.linspace(-yhalf, yhalf, 100)
normfit_dvy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_sum)*2*yhalf/nBins

%matplotlib inline

fig = plt.figure(figsize=(10,5),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,2,1)
ax1.hist(dvx_sum,bins=nBins)
ax1.plot(xval,normfit_dvx,'k')
ax1.text(-0.015,70,"$\mu$=%s"%("%.4f"%(mu_dvx)),fontsize=15)
ax1.text(-0.015,60,"$\sigma$=%s"%("%.4f"%(std_dvx)),fontsize=15)
#ax1.text(-0.052,12000,'(a)',fontsize=20)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)

ax2 = fig.add_subplot(1,2,2)
ax2.hist(dvy_sum,bins=nBins)
ax2.plot(yval,normfit_dvy,'k')
ax2.text(-0.01,70,"$\mu$=%s"%("%.4f"%(mu_dvy)),fontsize=15)
ax2.text(-0.01,60,"$\sigma$=%s"%("%.4f"%(std_dvy)),fontsize=15)
#ax2.text(-0.053,11500,'(b)',fontsize=20)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)

fig.tight_layout()
plt.show()


# plot both the one time result and all time result

In [ ]:
nBins = 30

dvx_raw = vxnoise[this_t,:]
dvy_raw = vynoise[this_t,:]
dvx_sum = np.array(dvx_list)
dvy_sum = np.array(dvy_list)

mu_dvx,std_dvx = norm.fit(dvx_sum)
xhalf = 0.025
xval = np.linspace(-xhalf, xhalf, 100)
normfit_dvx = norm.pdf(xval,mu_dvx,std_dvx)*len(dvx_sum)*2*xhalf/nBins

mu_dvy,std_dvy = norm.fit(dvy_sum)
yhalf = 0.025
yval = np.linspace(-yhalf, yhalf, 100)
normfit_dvy = norm.pdf(yval,mu_dvy,std_dvy)*len(dvy_sum)*2*yhalf/nBins

nBins_all = 100

dvx_raw_all = vxnoise.reshape(N*nrec)
dvy_raw_all = vynoise.reshape(N*nrec)
dvx_sum_all = np.array(dvx_list_all)
dvy_sum_all = np.array(dvy_list_all)

mu_dvx_all,std_dvx_all = norm.fit(dvx_sum_all)
xhalf_all = 0.039
xval_all = np.linspace(-xhalf_all, xhalf_all, 100)
normfit_dvx_all = norm.pdf(xval_all,mu_dvx_all,std_dvx_all)*len(dvx_sum_all)*2*xhalf_all/nBins_all

mu_dvy_all,std_dvy_all = norm.fit(dvy_sum_all)
yhalf_all = 0.038
yval_all = np.linspace(-yhalf_all, yhalf_all, 100)
normfit_dvy_all = norm.pdf(yval_all,mu_dvy_all,std_dvy_all)*len(dvy_sum_all)*2*yhalf_all/nBins_all

In [ ]:
# plots used in Supplemental Material

%matplotlib inline

fig = plt.figure(figsize=(16,8),facecolor=(1,1,1))

ax1 = fig.add_subplot(2,4,1)
ax1.hist(dvx_raw,bins=nBins)
ax1.text(-0.15,159,'(a)',fontsize=20)
ax1.set_xlabel('$\delta v_{x}$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)

ax2 = fig.add_subplot(2,4,2)
ax2.hist(dvy_raw,bins=nBins)
ax2.text(-0.135,152,'(b)',fontsize=20)
ax2.set_xlabel('$\delta v_{y}$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)

ax3 = fig.add_subplot(2,4,3)
ax3.hist(dvx_sum,bins=nBins)
ax3.plot(xval,normfit_dvx,'k')
ax3.text(-0.025,62,"$\mu$=%s"%("%.4f"%(mu_dvx)),fontsize=15)
ax3.text(-0.025,57,"$\sigma$=%s"%("%.4f"%(std_dvx)),fontsize=15)
ax3.text(-0.041,65,'(c)',fontsize=20)
ax3.set_xlabel('$\delta v_{x}$',fontsize=20)
ax3.set_ylabel('counts',fontsize=20)
ax3.locator_params(axis='x',nbins=4)

ax4 = fig.add_subplot(2,4,4)
ax4.hist(dvy_sum,bins=nBins)
ax4.plot(yval,normfit_dvy,'k')
ax4.text(-0.025,60,"$\mu$=%s"%("%.4f"%(mu_dvy)),fontsize=15)
ax4.text(-0.025,55,"$\sigma$=%s"%("%.4f"%(std_dvy)),fontsize=15)
ax4.text(-0.041,62.5,'(d)',fontsize=20)
ax4.set_xlabel('$\delta v_{y}$',fontsize=20)
ax4.set_ylabel('counts',fontsize=20)
ax4.locator_params(axis='x',nbins=5)

ax5 = fig.add_subplot(2,4,5)
ax5.hist(dvx_raw_all,bins=nBins_all)
ax5.text(-0.225,39000,'(e)',fontsize=20)
ax5.set_xlabel('$\delta v_{x}$',fontsize=20)
ax5.set_ylabel('counts',fontsize=20)

ax6 = fig.add_subplot(2,4,6)
ax6.hist(dvy_raw_all,bins=nBins_all)
ax6.text(-0.22,34100,'(f)',fontsize=20)
ax6.set_xlabel('$\delta v_{y}$',fontsize=20)
ax6.set_ylabel('counts',fontsize=20)

ax7 = fig.add_subplot(2,4,7)
ax7.hist(dvx_sum_all,bins=nBins_all)
ax7.plot(xval_all,normfit_dvx_all,'k')
ax7.text(-0.04,10000,"$\mu$=%s"%("%.4f"%(mu_dvx_all)),fontsize=15)
ax7.text(-0.04,9000,"$\sigma$=%s"%("%.4f"%(std_dvx_all)),fontsize=15)
ax7.text(-0.065,11000,'(g)',fontsize=20)
ax7.set_xlabel('$\delta v_{x}$',fontsize=20)
ax7.set_ylabel('counts',fontsize=20)

ax8 = fig.add_subplot(2,4,8)
ax8.hist(dvy_sum_all,bins=nBins_all)
ax8.plot(yval_all,normfit_dvy_all,'k')
ax8.text(-0.04,10000,"$\mu$=%s"%("%.4f"%(mu_dvy_all)),fontsize=15)
ax8.text(-0.04,9000,"$\sigma$=%s"%("%.4f"%(std_dvy_all)),fontsize=15)
ax8.text(-0.065,10600,'(h)',fontsize=20)
ax8.set_xlabel('$\delta v_{y}$',fontsize=20)
ax8.set_ylabel('counts',fontsize=20)

fig.tight_layout()
plt.show()
#fig.savefig('dv_hist_f006_w072twopi_ve0007sqrt20.pdf',dpi=fig.dpi,bbox_inches='tight')

# test with randomly generated numbers

In [ ]:
# test the statistics of randomly generated numbers that are meant to represent the noise in self-circling term
# get the v statistics with different time intervals for velocity calculation

test_time = 1
test_interval = 1
test_N = 10000
testx = np.zeros(test_interval*test_time*test_N)
testx_interval = np.zeros(test_time*test_N)
test_theta0 = np.random.uniform(0.0,1.0,test_N)*2*np.pi
omega = 1.0

for n in range(test_N):
    for t in range(test_time):
        testv = 0.0
        for ti in range(test_interval):
            v0 = np.random.normal(0.0,0.1)
            testx[ti + t*test_interval + test_interval*test_time*n] = v0*np.cos((ti + t*test_interval)*omega + test_theta0[n])
            testv += v0*np.cos((ti + t*test_interval)*omega + test_theta0[n])
        testx_interval[t+test_time*n] = testv*(1/test_interval)
        

In [ ]:
#test_int1 = testx_interval[-test_N:]
#test_int10 = testx_interval[-test_N:]
#test_int20 = testx_interval[-test_N:]
test_int1 = testx_interval
#test_int10 = testx_interval
#test_int20 = testx_interval

In [ ]:
mu_int10,std_int10 = norm.fit(test_int10)
xhalf_int10 = 0.08
xval_int10 = np.linspace(-xhalf_int10, xhalf_int10, 100)
normfit_int10 = norm.pdf(xval_int10,mu_int10,std_int10)*len(test_int10)*2*xhalf_int10/100

mu_int20,std_int20 = norm.fit(test_int20)
xhalf_int20 = 0.065
xval_int20 = np.linspace(-xhalf_int20, xhalf_int20, 100)
normfit_int20 = norm.pdf(xval_int20,mu_int20,std_int20)*len(test_int20)*2*xhalf_int20/100

%matplotlib inline

fig = plt.figure(figsize=(12,4),facecolor=(1,1,1))

ax1 = fig.add_subplot(1,3,1)
ax1.hist(test_int1,bins=100)
ax1.text(-0.51,825,'(a)',fontsize=20)
ax1.set_xlabel('$\delta$',fontsize=20)
ax1.set_ylabel('counts',fontsize=20)
ax1.tick_params(labelsize=12)

ax2 = fig.add_subplot(1,3,2)
ax2.hist(test_int10,bins=100)
ax2.plot(xval_int10,normfit_int10,'k')
ax2.text(-0.08,300,"$\mu$=%s"%("%.3f"%(-mu_int10)),fontsize=15)
ax2.text(-0.08,270,"$\sigma$=%s"%("%.3f"%(std_int10)),fontsize=15)
ax2.text(-0.13,322,'(b)',fontsize=20)
ax2.set_xlabel('$\delta$',fontsize=20)
ax2.set_ylabel('counts',fontsize=20)
ax2.tick_params(labelsize=12)

ax3 = fig.add_subplot(1,3,3)
ax3.hist(test_int20,bins=100)
ax3.plot(xval_int20,normfit_int20,'k')
ax3.text(-0.06,325,"$\mu$=%s"%("%.3f"%(-mu_int20)),fontsize=15)
ax3.text(-0.06,295,"$\sigma$=%s"%("%.3f"%(std_int20)),fontsize=15)
ax3.text(-0.103,352,'(c)',fontsize=20)
ax3.set_xlabel('$\delta$',fontsize=20)
ax3.set_ylabel('counts',fontsize=20)
ax3.tick_params(labelsize=12)

fig.tight_layout()
plt.show()
#fig.savefig('rv_window_test.pdf',dpi=fig.dpi,bbox_inches='tight')